# Sygaldry Agent Interactive Testing Notebook

This notebook provides an interactive environment to test and explore Sygaldry agents.

## Setup

First, let's set up the environment and install dependencies.

In [ ]:
# Install required packages
!pip install mirascope>=2.0.0a1 pydantic>=2.0.0 rich>=13.0.0 python-dotenv

In [ ]:
# Import necessary libraries
import sys
import os
from pathlib import Path
import json
from datetime import datetime
from typing import Any, Dict, List, Optional
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

# Add the packages directory to the path (from examples directory)
sys.path.insert(0, str(Path.cwd().parent / "packages" / "sygaldry_registry"))

print("✅ Setup complete!")

## Configure API Keys

Set your API keys here. You need at least one to run the agents.

In [ ]:
# Option 1: Set API keys directly (not recommended for shared notebooks)
# os.environ["OPENAI_API_KEY"] = "your-openai-key-here"
# os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-key-here"

# Option 2: Load from .env file (recommended)
from dotenv import load_dotenv
load_dotenv()

# Check which keys are available
available_keys = []
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY"]:
    if os.getenv(key):
        available_keys.append(key.replace("_API_KEY", ""))

if available_keys:
    print(f"✅ API keys configured for: {', '.join(available_keys)}")
else:
    print("❌ No API keys found. Please set at least one API key.")

## Helper Functions

Utility functions to display results nicely in the notebook.

In [ ]:
def display_result(result: Any, title: str = "Result"):
    """Display agent results in a nice format."""
    html = f"<h3>{title}</h3>"
    
    if hasattr(result, "model_dump"):
        # Pydantic model
        data = result.model_dump()
        html += "<pre style='background-color: #f0f0f0; padding: 10px; border-radius: 5px;'>"
        html += json.dumps(data, indent=2)
        html += "</pre>"
    elif isinstance(result, dict):
        html += "<pre style='background-color: #f0f0f0; padding: 10px; border-radius: 5px;'>"
        html += json.dumps(result, indent=2)
        html += "</pre>"
    elif isinstance(result, str):
        html += f"<div style='background-color: #e8f4ea; padding: 10px; border-radius: 5px; border-left: 3px solid #4CAF50;'>{result}</div>"
    else:
        html += f"<div>{str(result)}</div>"
    
    display(HTML(html))

def display_error(error: Exception):
    """Display errors nicely."""
    html = f"<div style='background-color: #fee; padding: 10px; border-radius: 5px; border-left: 3px solid #f44336;'>"
    html += f"<strong>Error:</strong> {str(error)}"
    html += "</div>"
    display(HTML(html))

print("✅ Helper functions loaded!")

## 1. Text Summarization Agent

Let's start with the text summarization agent. It can create summaries in different styles.

In [ ]:
from components.agents.text_summarization import (
    summarize_text,
    quick_summary,
    executive_brief,
    multi_style_summary
)

# Sample text
sample_text = """
Artificial Intelligence (AI) has become one of the most transformative technologies of the 21st century.
From its early beginnings in the 1950s, when computer scientists first began to explore the possibility
of creating machines that could think and learn like humans, AI has evolved dramatically. Today, AI
systems power everything from voice assistants on our phones to complex decision-making systems in
healthcare and finance. Machine learning, a subset of AI, has particularly revolutionized how we
process and analyze large datasets, enabling computers to identify patterns and make predictions with
unprecedented accuracy. Deep learning, inspired by the human brain's neural networks, has pushed the
boundaries even further, achieving remarkable results in image recognition, natural language processing,
and game playing. As we look to the future, AI continues to present both immense opportunities and
significant challenges, from ethical considerations about bias and transparency to questions about
the impact on employment and society as a whole.
"""

print("Text Summarization Agent loaded!")
print(f"Sample text length: {len(sample_text.split())} words")

In [ ]:
# Quick summary
try:
    result = quick_summary(sample_text)
    display_result(result, "Quick Summary")
except Exception as e:
    display_error(e)

In [ ]:
# Technical summary with more detail
try:
    result = summarize_text(sample_text, style="technical", max_length=100)
    display_result(result, "Technical Summary")
except Exception as e:
    display_error(e)

In [ ]:
# Executive brief
try:
    result = executive_brief(sample_text)
    display_result(result, "Executive Brief")
except Exception as e:
    display_error(e)

## 2. Sentiment Analysis Agent

Analyze sentiment, emotions, and subjectivity in text.

In [ ]:
from components.agents.sentiment_analysis import analyze_sentiment

# Test different sentiments
test_texts = [
    "I absolutely love this product! It exceeded all my expectations and the customer service was fantastic.",
    "This is the worst experience I've ever had. Completely disappointed and will never buy again.",
    "The product is okay. It works as described but nothing special. Average quality for the price.",
    "I'm worried about the environmental impact, but excited about the innovation potential."
]

print("Sentiment Analysis Agent loaded!")

In [ ]:
# Analyze each text
for i, text in enumerate(test_texts, 1):
    print(f"\nText {i}: {text[:50]}...")
    try:
        result = analyze_sentiment(text)
        display_result(result, f"Sentiment Analysis {i}")
    except Exception as e:
        display_error(e)

## 3. PII Scrubbing Agent

Remove personally identifiable information from text.

In [ ]:
from components.agents.pii_scrubbing import scrub_pii

# Text with PII
pii_text = """
Hello, my name is John Smith and I work at Acme Corporation.
You can reach me at john.smith@example.com or call me at 555-123-4567.
My social security number is 123-45-6789 and I live at 123 Main St, Anytown, USA 12345.
My credit card number is 4532-1234-5678-9012 and expires on 12/25.
"""

print("PII Scrubbing Agent loaded!")
print("\nOriginal text with PII:")
print(pii_text)

In [ ]:
# Mask PII
try:
    result = scrub_pii(pii_text, scrubbing_method="mask")
    display_result(result, "Masked PII")
except Exception as e:
    display_error(e)

In [ ]:
# Redact PII
try:
    result = scrub_pii(pii_text, scrubbing_method="redact")
    display_result(result, "Redacted PII")
except Exception as e:
    display_error(e)

## 4. Code Review Agent

Review code for security issues, best practices, and improvements.

In [ ]:
from components.agents.code_review import review_code

# Sample code with issues
code_to_review = """
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total = total + n
    average = total / len(numbers)  # Potential ZeroDivisionError
    return average

def get_user_data(user_id):
    import sqlite3
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    # SQL Injection vulnerability!
    query = f"SELECT * FROM users WHERE id = {user_id}"
    cursor.execute(query)
    result = cursor.fetchone()
    # Connection not closed
    return result

def process_password(password):
    # Storing password in plain text!
    with open('passwords.txt', 'a') as f:
        f.write(password + '\n')
    return password
"""

print("Code Review Agent loaded!")
print("\nCode to review:")
print(code_to_review)

In [ ]:
# Perform code review
try:
    result = review_code(
        code=code_to_review,
        language="python",
        focus_areas=["security", "performance", "style", "best_practices"]
    )
    display_result(result, "Code Review Results")
except Exception as e:
    display_error(e)

## 5. Bug Triage Agent

Analyze and classify bug reports.

In [ ]:
from components.agents.bug_triage import triage_bug

# Sample bug report
bug_report = {
    "title": "Application crashes when uploading large files",
    "description": """
    When I try to upload a file larger than 100MB, the application crashes with an out of memory error.
    This happens consistently on the production server but not in development.
    
    Steps to reproduce:
    1. Go to the upload page
    2. Select a file larger than 100MB
    3. Click upload
    4. Application crashes after a few seconds
    
    Error message: "JavaScript heap out of memory"
    """,
    "environment": "Production server, Node.js 16.x, 2GB RAM",
    "user_impact": "high",
    "frequency": "always"
}

print("Bug Triage Agent loaded!")

In [ ]:
# Triage the bug
try:
    result = triage_bug(**bug_report)
    display_result(result, "Bug Triage Results")
except Exception as e:
    display_error(e)

## 6. Task Prioritization Agent

Prioritize tasks using the Eisenhower matrix.

In [ ]:
from components.agents.task_prioritization import prioritize_tasks

# List of tasks to prioritize
tasks = [
    "Fix critical production bug causing data loss",
    "Update project documentation",
    "Attend weekly team meeting",
    "Research new framework for next quarter",
    "Respond to customer support emails",
    "Refactor legacy code module",
    "Prepare presentation for tomorrow's demo",
    "Review pull requests from team members",
    "Set up monitoring for new microservice",
    "Learn new programming language"
]

print("Task Prioritization Agent loaded!")
print(f"\nTasks to prioritize ({len(tasks)} tasks):")
for i, task in enumerate(tasks, 1):
    print(f"{i}. {task}")

In [ ]:
# Prioritize tasks
try:
    result = prioritize_tasks(tasks)
    display_result(result, "Task Prioritization Results")
except Exception as e:
    display_error(e)

## 7. Contract Analysis Agent

Analyze legal documents for risks and key terms.

In [ ]:
from components.agents.contract_analysis import analyze_contract

# Sample contract text
contract_text = """
SERVICE AGREEMENT

This Service Agreement ("Agreement") is entered into as of January 1, 2024, between
ABC Corporation ("Client") and XYZ Services Inc. ("Service Provider").

1. SERVICES
Service Provider agrees to provide software development services as outlined in Exhibit A.

2. PAYMENT TERMS
Client shall pay Service Provider $10,000 per month, due within 30 days of invoice.
Late payments will incur a 1.5% monthly interest charge.

3. TERM AND TERMINATION
This Agreement shall continue for 12 months. Either party may terminate with 30 days written notice.
Early termination by Client requires payment of 3 months' fees as a penalty.

4. INTELLECTUAL PROPERTY
All work product created under this Agreement shall be the sole property of the Client.

5. LIABILITY LIMITATION
Service Provider's total liability shall not exceed the fees paid in the last 6 months.
Service Provider is not liable for any indirect, consequential, or punitive damages.

6. CONFIDENTIALITY
Both parties agree to maintain confidentiality of proprietary information for 5 years.
"""

print("Contract Analysis Agent loaded!")

In [ ]:
# Analyze the contract
try:
    result = analyze_contract(
        contract_text=contract_text,
        contract_type="service",
        focus_areas=["payment_terms", "termination", "liability", "intellectual_property"]
    )
    display_result(result, "Contract Analysis Results")
except Exception as e:
    display_error(e)

## Custom Agent Testing

Use this section to test any other agent from the Sygaldry registry.

In [ ]:
# List all available agents
import os
agents_dir = Path.cwd().parent / "packages" / "sygaldry_registry" / "components" / "agents"
if agents_dir.exists():
    agents = [d.name for d in agents_dir.iterdir() if d.is_dir() and not d.name.startswith('__')]
    print("Available agents:")
    for i, agent in enumerate(sorted(agents), 1):
        print(f"{i:2d}. {agent}")
else:
    print("Agents directory not found")

In [ ]:
# Import and test a specific agent
# Example: Financial Statement Analyzer

try:
    from components.agents.financial_statement_analyzer import analyze_financial_statement
    
    # Sample financial data
    financial_data = """
    Q3 2024 Financial Summary:
    - Revenue: $15.2 million (up 23% YoY)
    - Operating Expenses: $12.1 million
    - Net Income: $2.3 million
    - Total Assets: $45 million
    - Total Liabilities: $18 million
    - Cash and Cash Equivalents: $8.5 million
    """
    
    result = analyze_financial_statement(
        statement_text=financial_data,
        statement_type="quarterly_report",
        analysis_depth="comprehensive"
    )
    
    display_result(result, "Financial Analysis Results")
    
except Exception as e:
    display_error(e)

## Interactive Testing Section

Use this cell to interactively test any agent with custom inputs.

In [ ]:
# Interactive testing - modify as needed

# Choose your agent
agent_name = "text_summarization"  # Change this to any agent name

# Your custom input
your_text = """
Enter your custom text here for testing...
"""

# Import and test
try:
    # Example for text summarization
    if agent_name == "text_summarization":
        from components.agents.text_summarization import summarize_text
        result = summarize_text(your_text, style="executive", max_length=100)
        display_result(result, "Custom Test Result")
    
    # Add more agent cases as needed
    # elif agent_name == "another_agent":
    #     from components.agents.another_agent import function_name
    #     result = function_name(your_input)
    #     display_result(result, "Custom Test Result")
    
except Exception as e:
    display_error(e)

## Summary

You've now tested several Sygaldry agents! Here's what you can do next:

1. **Explore More Agents**: Import and test other agents from the registry
2. **Custom Inputs**: Modify the examples above with your own data
3. **Combine Agents**: Chain multiple agents together for complex workflows
4. **Build Applications**: Integrate these agents into your applications

For more information, check the documentation in:
- Individual agent READMEs: `packages/sygaldry_registry/components/agents/*/README.md`
- Main project README: `README.md`
- Development guidelines: `CLAUDE.md`